# Lekcija 18: Prostori boja

U prethodnim smo lekcijama slike u boji pretvarali u sive tonove i nastavili s obradom. Ova lekcija razmatra što ta pretvorba zapravo radi te druge načine prikaza boje osim RGB-a, od kojih je svaki prikladan za drukčiji zadatak: **HSV**, koji odvaja boju od svjetline, **YCbCr**, koji odvaja luminancijsku od kromatskih komponenti i temelj je kompresije videa iz Lekcije 17, te **L\*a\*b\***, oblikovan tako da udaljenosti odgovaraju ljudskoj percepciji.


In [ ]:
import numpy as np
import cv2
import matplotlib.pyplot as plt

## Sivi tonovi već predstavljaju odabir prostora boja

Najjednostavniji način pretvaranja boje u sive tonove jest usrednjavanje triju kanala:

$$Y = (R + G + B)/3$$

Iako to donekle radi, ljudski vidni sustav osjetljiviji je na zelenu nego na ostale boje. Zato je bolje zelenoj dati veću težinu:

$$Y = (R + 2G + B)/4$$

Još bolji način jest koristiti težine dobivene stvarnim pokusima s ljudskim ispitanicima, što vodi do poznate formule koju koristi `cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)`:

$$Y = 0.299 R + 0.587 G + 0.114 B$$

Te težine proizlaze iz osjetljivosti ljudskog vida na luminanciju: oko je znatno osjetljivije na zelenu nego na plavu, pa se zelena i plava jednakog intenziteta *ne bi trebale* preslikati u istu razinu sivog tona, iako bi ih jednostavan prosjek kanala tretirao jednako.


### Zašto je ta formula zastarjela, iako je široko korištena i još korisna

U toj se formuli kriju dvije stvari koje je korisno poznavati:

1. **Stare primarne boje.** Težine $0.299, 0.587, 0.114$ potječu iz standarda NTSC iz 1953. (Rec. 601), izvedenog iz konkretnih crvenih, zelenih i plavih fosfora ranih televizora u boji s katodnom cijevi. Moderni LED zasloni, kao i gotovo sve ostalo proizvedeno od ranih 2000-ih, koriste zasićenije primarne boje, pa su kolorimetrijski ispravne težine za moderni zaslon zapravo $Y = 0.2126R + 0.7152G + 0.0722B$ (Rec. 709/sRGB): daju primjetno *veću* težinu zelenoj i *manju* crvenoj. Funkcija `cv2.cvtColor(..., COLOR_BGR2GRAY)` i dalje koristi stare težine Rec. 601, uglavnom radi kompatibilnosti s desetljećima postojećeg koda.
2. **Pretpostavka linearne svjetlosti.** Formula ponderiranog zbroja daje stvarnu luminanciju samo kada su $R$, $G$ i $B$ *linearne* vrijednosti svjetlosti. Međutim, kao što ćemo vidjeti u Lekciji 26, pohranjene vrijednosti piksela gama su *kodirane*, a ne linearne. Funkcija `cv2.cvtColor` primjenjuje težine izravno na izvorne kodirane 8-bitne vrijednosti, bez prethodnog dekodiranja game &mdash; brzo i dovoljno približno za većinu zadataka računalnog vida, ali ne daje fizikalno ispravnu luminanciju potrebnu za kalibraciju zaslona ili fotometriju. Kako biste vidjeli problem, izradite čisto plavu sliku s RGB=(0,0,255) i pretvorite je pomoću OpenCV-a: svugdje ćete dobiti vrijednost 29, koja će pogrešno izgledati gotovo potpuno tamno, iako je potpuno zasićena plava oku razmjerno svijetla boja.


In [ ]:
weights_601 = np.array([0.114, 0.587, 0.299])   # BGR order, what cv2 actually uses (Rec. 601)
weights_709 = np.array([0.0722, 0.7152, 0.2126])  # BGR order, modern primaries (Rec. 709 / sRGB)

for name, bgr in [('pure red', (0, 0, 255)), ('pure green', (0, 255, 0))]:
    bgr = np.array(bgr, dtype=np.float64)
    y_601 = bgr @ weights_601
    y_709 = bgr @ weights_709
    print(f'{name:>10}: Rec. 601 (cv2) Y = {y_601:6.1f}    Rec. 709 (modern) Y = {y_709:6.1f}')

In [ ]:
pure_green = np.full((10, 10, 3), (0, 255, 0), dtype=np.uint8)   # BGR
pure_blue = np.full((10, 10, 3), (255, 0, 0), dtype=np.uint8)

for name, patch in [('green', pure_green), ('blue', pure_blue)]:
    weighted = cv2.cvtColor(patch, cv2.COLOR_BGR2GRAY)[0, 0]
    naive = patch[0, 0].astype(np.float64).mean()
    print(f'{name:>6}: cv2 (luminance-weighted) gray = {weighted:>3},   naive average = {naive:.0f}')

Jednostavan prosjek uopće ne razlikuje zelenu i plavu jer obje daju 85; perceptivno ponderirana inačica ispravno daje znatno svjetliju zelenu, 150, od plave, 29, pri istom intenzitetu kanala.


## RGB kanali snažno su korelirani

U većini stvarnih slika prevladavaju promjene osvjetljenja i zasjenjenja: površina u cjelini postaje svjetlija ili tamnija, pri čemu se sva tri kanala zajedno skaliraju. Zato su R, G i B međusobno snažno korelirani &mdash; to nije posebno učinkovit ni praktičan način razdvajanja pitanja „koja je ovo boja?” od pitanja „koliko je jako osvijetljena?”.


In [ ]:
rng = np.random.default_rng(0)
shading = rng.uniform(0.3, 1.0, (150, 150))  # a smoothly varying "lighting" field
shaded_img = np.stack([shading * 200, shading * 150, shading * 100], axis=-1).astype(np.uint8)  # BGR

shaded_img = cv2.imread('../img/sheepdog.jpg')
r = shaded_img[:, :, 2].ravel().astype(np.float64)
g = shaded_img[:, :, 1].ravel().astype(np.float64)
print(f'correlation between R and G channels: {np.corrcoef(r, g)[0, 1]:.4f}')

img_h, img_w = shaded_img.shape[:2]
fig, axes = plt.subplots(1, 2, figsize=(4 * img_w / img_h + 4, 4),
                          gridspec_kw={'width_ratios': [img_w / img_h, 1]})
axes[0].imshow(cv2.cvtColor(shaded_img, cv2.COLOR_BGR2RGB))
axes[0].set_title('Image')
axes[0].axis('off')
axes[1].scatter(r, g, s=2, alpha=0.3)
axes[1].set_xlabel('R'); axes[1].set_ylabel('G')
axes[1].set_title('R and G are highly correlated')
axes[1].set_xlim(0, 255)
axes[1].set_ylim(0, 255)
axes[1].set_aspect('equal', adjustable='box')
plt.show()

## HSV: odvajanje boje od svjetline

HSV (Hue, Saturation, Value, odnosno ton, zasićenost i vrijednost) preparametrizira boju tako da ton opisuje *koja* je boja, neovisno o njezinoj svjetlini ili ispranosti, zasićenost opisuje koliko je boja izražena i čista, a vrijednost opisuje svjetlinu. Time segmentacija prema boji postaje znatno otpornija na promjene osvjetljenja nego izravno pragovanje u RGB-u.


In [ ]:
size = 200
flat_img = np.zeros((size, size, 3), dtype=np.uint8)
cv2.circle(flat_img, (100, 100), 70, (0, 100, 255), -1)  # an orange disk, BGR

yy, xx = np.mgrid[0:size, 0:size]
lighting_gradient = (0.25 + 0.9 * (xx / size))  # dark on the left, bright on the right
shaded_disk = np.clip(flat_img.astype(np.float64) * lighting_gradient[..., None], 0, 255).astype(np.uint8)

hsv = cv2.cvtColor(shaded_disk, cv2.COLOR_BGR2HSV)
hue = hsv[:, :, 0]

mask_hsv = (hue > 5) & (hue < 25)                                              # hue range only
mask_rgb = ((shaded_disk[:, :, 2] > 150) & (shaded_disk[:, :, 1] > 50) &
            (shaded_disk[:, :, 1] < 180) & (shaded_disk[:, :, 0] < 80))        # fixed RGB range
true_mask = (xx - 100)**2 + (yy - 100)**2 <= 70**2

def iou(a, b):
    return (a & b).sum() / (a | b).sum()

print(f'IoU vs. true disk, HSV hue threshold: {iou(mask_hsv, true_mask):.3f}')
print(f'IoU vs. true disk, fixed RGB range:   {iou(mask_rgb, true_mask):.3f}')

fig, axes = plt.subplots(1, 3, figsize=(9, 3.5))
axes[0].imshow(cv2.cvtColor(shaded_disk, cv2.COLOR_BGR2RGB))
axes[0].set_title('Shaded disk (left dark, right bright)')
axes[1].imshow(mask_hsv, cmap='gray')
axes[1].set_title('HSV hue threshold')
axes[2].imshow(mask_rgb, cmap='gray')
axes[2].set_title('Fixed RGB range')
for ax in axes:
    ax.axis('off')
plt.tight_layout()
plt.show()

Pragovanje tona savršeno izdvaja disk bez obzira na promjenu osvjetljenja preko njega, dok fiksni RGB raspon potpuno propušta zatamnjenu stranu &mdash; upravo slabost segmentacije boje u čistom RGB-u koja se pojavljuje čim osvjetljenje nije potpuno jednoliko.


## YCbCr: luma i kroma te zašto video jače komprimira boju

YCbCr, koji se interno koristi u JPEG-u i gotovo svim videokodecima, rastavlja sliku na **lumu** ($Y$, približno svjetlinu) i dva kanala **krome** ($C_b, C_r$, približno opis „koliko je plavo” i „koliko je crveno”). Ljudski vidni sustav znatno bolje razlučuje fine prostorne detalje lume nego krome &mdash; biološka osnova **poduzorkovanja krome**, odnosno pohrane krome na nižoj razlučivosti od lume, primjerice u uobičajenom videoformatu „4:2:0”. To je jedno od povoljnih smanjenja količine podataka u JPEG postupku iz Lekcije 17.


In [ ]:
textured = np.zeros((200, 200, 3), dtype=np.uint8)
cv2.rectangle(textured, (20, 20), (90, 180), (255, 120, 30), -1)
cv2.circle(textured, (140, 100), 50, (30, 200, 255), -1)
gradient_x = np.mgrid[0:200, 0:200][1]
textured[:, :, 0] = np.clip(textured[:, :, 0].astype(int) + (gradient_x // 4) % 50, 0, 255)
textured = np.clip(textured.astype(np.float64) + rng.normal(0, 5, textured.shape), 0, 255).astype(np.uint8)

Y, Cr, Cb = cv2.split(cv2.cvtColor(textured, cv2.COLOR_BGR2YCrCb))

def subsample_then_upsample(channel, factor=8):
    small = cv2.resize(channel, (channel.shape[1] // factor, channel.shape[0] // factor), interpolation=cv2.INTER_AREA)
    return cv2.resize(small, (channel.shape[1], channel.shape[0]), interpolation=cv2.INTER_LINEAR)

chroma_degraded = cv2.cvtColor(cv2.merge([Y, subsample_then_upsample(Cr), subsample_then_upsample(Cb)]),
                                cv2.COLOR_YCrCb2BGR)
luma_degraded = cv2.cvtColor(cv2.merge([subsample_then_upsample(Y), Cr, Cb]), cv2.COLOR_YCrCb2BGR)

error_chroma = np.abs(chroma_degraded.astype(int) - textured.astype(int)).mean()
error_luma = np.abs(luma_degraded.astype(int) - textured.astype(int)).mean()
print(f'mean abs error, chroma degraded 8x: {error_chroma:.2f}')
print(f'mean abs error, luma degraded 8x:   {error_luma:.2f}')

fig, axes = plt.subplots(1, 3, figsize=(9, 3.5))
for ax, im, title in zip(axes, [textured, chroma_degraded, luma_degraded],
                          ['Original', 'Chroma subsampled 8x\n(less visible loss)', 'Luma subsampled 8x\n(more visible loss)']):
    ax.imshow(cv2.cvtColor(im, cv2.COLOR_BGR2RGB))
    ax.set_title(title, fontsize=9)
    ax.axis('off')
plt.tight_layout()
plt.show()

Jednako smanjenje razlučivosti daje primjetno manju pogrešku kod krome nego kod lume &mdash; isti gubitak detalja manje smeta kada se odnosi na boju nego na svjetlinu, što je upravo kompromis koji iskorištava poduzorkovanje krome.


## L\*a\*b\*: perceptivno ujednačen prostor

Euklidska udaljenost u RGB-u slabo opisuje koliko dvije boje stvarno *izgledaju* različito: jednake RGB udaljenosti mogu odgovarati vrlo različitim perceptivnim razlikama, ovisno o položaju u prostoru boja. CIELAB je posebno oblikovan tako da euklidska udaljenost između dviju L\*a\*b\* boja znatno dosljednije aproksimira opaženu razliku &mdash; korisno za metrike kvalitete temeljene na boji, grupiranje i usporedbu.


In [ ]:
target_rgb_distance = 30.0
n_samples = 2000

c1 = rng.uniform(0, 255, (n_samples, 3))
direction = rng.normal(size=(n_samples, 3))
direction /= np.linalg.norm(direction, axis=1, keepdims=True)
c2 = c1 + direction * target_rgb_distance

valid = np.all((c2 >= 0) & (c2 <= 255), axis=1)
c1, c2 = c1[valid].astype(np.uint8), c2[valid].astype(np.uint8)

lab1 = cv2.cvtColor(c1.reshape(1, -1, 3), cv2.COLOR_BGR2LAB).reshape(-1, 3).astype(np.float64)
lab2 = cv2.cvtColor(c2.reshape(1, -1, 3), cv2.COLOR_BGR2LAB).reshape(-1, 3).astype(np.float64)
lab_distances = np.linalg.norm(lab1 - lab2, axis=1)

print(f'RGB distance held fixed at exactly {target_rgb_distance}')
print(f'resulting L*a*b* distances: min={lab_distances.min():.1f}, '
      f'median={np.median(lab_distances):.1f}, max={lab_distances.max():.1f}')

plt.hist(lab_distances, bins=40)
plt.xlabel('L*a*b* distance')
plt.ylabel('count')
plt.title(f'Perceptual (L*a*b*) distance for {n_samples} color pairs,\nall with the SAME RGB distance ({target_rgb_distance})')
plt.show()

Svaki od ovih parova ima *točno* istu RGB udaljenost &mdash; ipak, odgovarajuća perceptivna razlika u L\*a\*b\* prostoru kreće se od jedva primjetne do više od 15 puta veće, ovisno samo o *položaju* para u prostoru boja. Svaki algoritam koji izravnu RGB udaljenost koristi kao zamjenu za pitanje „koliko ove boje izgledaju slično?”, poput uspoređivanja boja najbližim susjedom, k-means grupiranja boja ili pragova oduzimanja pozadine, nasljeđuje isto izobličenje. Prelazak na Lab udaljenost jednostavno je, standardno rješenje.


### Zadaci

1. Ponovite usporedbu HSV i RGB segmentacije s gradijentom zasjenjenja koji malo mijenja i ton, primjerice miješanjem prema drugoj boji na jednom rubu umjesto samo promjene svjetline. Radi li pragovanje samo prema tonu i dalje jednako dobro?
2. Povećajte faktor poduzorkovanja krome i lume s 8 na 20. Vrijedi li i dalje da je gubitak krome manje vidljiv od gubitka lume ili ta tvrdnja pri velikom faktoru prestaje vrijediti?
3. Pregledom `c1`/`c2` u Lab pokusu pronađite jedan par s vrlo *malom* Lab udaljenošću i jedan s vrlo *velikom* L\*a\*b\* udaljenošću, iako oba imaju istu RGB udaljenost. Prikažite četiri boje kao uzorke i svojim riječima opišite zašto par s malom Lab udaljenošću izgleda sličnije.
